# FeatureRanker: fantasy points and injury lines

## tl;dr

- `lag1_games` is the strongest exact feature for both targets and is top ten in all five component rankings.
- Points then favors prior-year production: receptions, passing touchdowns, receiving yards, receiving first downs, chunk receptions, and passing yards.
- Injury then favors exposure and receiving workload: yards after catch, receiving yards, receptions, offense snaps, and targets, followed by draft and age context.
- The strongest prior injury-history feature is `injury_lag2_injury_report_weeks` at consensus rank 44, so injury history is useful but secondary in this candidate pool.
- Results are discovery-only predictive associations. They are not causal effects, and the injury target is report presence rather than severity or games missed.


## Publication note

The displayed outputs were generated from the same repository feature pipeline on August 20, 2026. The path setup in this published copy was changed from machine-specific absolute paths to repository-relative discovery so the notebook can be rerun from any checkout.

## Context & Methods

The decision is which leakage-controlled inputs deserve priority in future fantasy-points and injury-report modeling. The primary reader is technical. Rankings use only discovery-period labels, leaving the project's retrospective audit seasons out of feature ranking.

### Key Assumptions

- “Points” means the primary pooled offense line (QB, RB, WR, TE), not the separately modeled low-confidence kicker line.
- Points use target seasons 2013-2021 and the selected Phase 2 stable core-plus-trajectory contract.
- Injury uses target seasons 2013-2021 and the complete Phase 3 August-origin contract.
- Numeric missing values are median-imputed from the discovery data only, with any all-missing columns filled with zero.
- `featureranker` 3.0.4 runs Random Forest, XGBoost, mutual information, ANOVA F-test, and L1 ranking. The primary consensus is automatic probe-weighted reciprocal-rank voting.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from threadpoolctl import threadpool_limits

import featureranker
from featureranker import feature_ranking, voting

REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "fantasy_football").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run the notebook from the fantasy-football repository tree.")

WORKSPACE = REPO_ROOT / "experiments" / "feature_ranker"
OUTPUT_DIR = WORKSPACE / "artifacts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO_ROOT))

RANDOM_STATE = 20260820
MAX_THREADS = 4
DISCOVERY_MIN_SEASON = 2013
DISCOVERY_MAX_SEASON = 2021
OFFENSE_POSITIONS = ("QB", "RB", "WR", "TE")

print({"featureranker": featureranker.__version__, "output_dir": str(OUTPUT_DIR)})

{'featureranker': '3.0.4', 'output_dir': 'C:\\Users\\lhall\\Documents\\Codex\\2026-08-20\\find-where-the-fantasy-football-project\\analysis\\outputs'}


## Data

### 1. Rebuild the locked points and injury feature matrices

Both matrices are reconstructed from the project’s frozen local inputs and feature-lineage code. No football data is downloaded.

In [2]:
from fantasy_football.phase2_august_runner import (
    _build_inputs,
    _candidate,
    load_august_plan,
    safe_feature_frame,
)
from fantasy_football.phase3_injury import build_injury_feature_bundle

points_plan_path = REPO_ROOT / "experiments" / "phase2" / "production" / "august9_plan_v1.json"
points_plan = load_august_plan(points_plan_path)
_, points_bundles, points_tables = _build_inputs(
    REPO_ROOT,
    points_plan,
    maximum_target_season=DISCOVERY_MAX_SEASON,
    needed_tiers={"stable"},
)
points_bundle = points_bundles["stable"]
points_table = points_tables["stable"]
points_candidate_record = next(
    record for record in points_plan["candidates"]
    if record["identifier"] == "off_stable_core_et_all"
)
points_candidate = _candidate(points_candidate_record)
points_features_all = safe_feature_frame(points_bundle, points_candidate, "offense")
points_mask = (
    points_table["target_season"].between(DISCOVERY_MIN_SEASON, DISCOVERY_MAX_SEASON)
    & points_table["model_position"].isin(OFFENSE_POSITIONS)
    & points_table["target_points"].notna()
)
points_X_raw = points_features_all.loc[points_mask].reset_index(drop=True)
points_y = points_table.loc[points_mask, "target_points"].astype("float64").reset_index(drop=True)
points_metadata = points_table.loc[
    points_mask, ["target_season", "player_id", "model_position"]
].reset_index(drop=True)

injury_directory = REPO_ROOT / "experiments" / "phase3" / "data" / "raw"
injury_bundle = build_injury_feature_bundle(REPO_ROOT, injury_directory)
injury_mask = injury_bundle.metadata["target_season"].between(
    DISCOVERY_MIN_SEASON, DISCOVERY_MAX_SEASON
)
injury_X_raw = injury_bundle.frame.loc[injury_mask].reset_index(drop=True)
injury_y = injury_bundle.labels.loc[
    injury_mask, "physical_injury_reported"
].astype("int8").reset_index(drop=True)
injury_metadata = injury_bundle.metadata.loc[
    injury_mask, ["target_season", "player_id", "model_position"]
].reset_index(drop=True)

print(
    {
        "points_shape": points_X_raw.shape,
        "injury_shape": injury_X_raw.shape,
        "points_seasons": [int(points_metadata.target_season.min()), int(points_metadata.target_season.max())],
        "injury_seasons": [int(injury_metadata.target_season.min()), int(injury_metadata.target_season.max())],
    }
)

{'points_shape': (8613, 228), 'injury_shape': (9077, 500), 'points_seasons': [2013, 2021], 'injury_seasons': [2013, 2021]}


### 2. Profile grain, completeness, and leakage controls

The checks below verify unique player-season-position rows, complete targets, finite post-imputation inputs, constant-column handling, and the target definitions used by each line.

In [3]:
def prepare_features(frame: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, object]]:
    numeric = frame.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    missing_rates = numeric.isna().mean()
    medians = numeric.median(axis=0)
    imputed = numeric.fillna(medians).fillna(0.0).astype("float32")
    constant_columns = imputed.columns[imputed.nunique(dropna=False).le(1)].tolist()
    usable = imputed.drop(columns=constant_columns)
    profile = {
        "rows": int(len(usable)),
        "input_features": int(frame.shape[1]),
        "ranked_features": int(usable.shape[1]),
        "constant_features_removed": constant_columns,
        "columns_with_missing_values": int((missing_rates > 0).sum()),
        "maximum_missing_rate": float(missing_rates.max()),
        "finite_after_imputation": bool(np.isfinite(usable.to_numpy()).all()),
    }
    return usable, profile


points_X, points_profile = prepare_features(points_X_raw)
injury_X, injury_profile = prepare_features(injury_X_raw)

quality = {
    "points": {
        **points_profile,
        "duplicate_keys": int(points_metadata.duplicated(["target_season", "player_id", "model_position"]).sum()),
        "missing_targets": int(points_y.isna().sum()),
        "target_min": float(points_y.min()),
        "target_median": float(points_y.median()),
        "target_max": float(points_y.max()),
        "zero_target_rate": float(points_y.eq(0).mean()),
        "positions": points_metadata["model_position"].value_counts().sort_index().to_dict(),
    },
    "injury": {
        **injury_profile,
        "duplicate_keys": int(injury_metadata.duplicated(["target_season", "player_id", "model_position"]).sum()),
        "missing_targets": int(injury_y.isna().sum()),
        "positive_rate": float(injury_y.mean()),
        "class_counts": injury_y.value_counts().sort_index().to_dict(),
        "positions": injury_metadata["model_position"].value_counts().sort_index().to_dict(),
    },
}

assert quality["points"]["duplicate_keys"] == 0
assert quality["injury"]["duplicate_keys"] == 0
assert quality["points"]["missing_targets"] == 0
assert quality["injury"]["missing_targets"] == 0
assert points_profile["finite_after_imputation"]
assert injury_profile["finite_after_imputation"]

(OUTPUT_DIR / "data_quality.json").write_text(json.dumps(quality, indent=2, default=int), encoding="utf-8")
quality

{'points': {'rows': 8613,
  'input_features': 228,
  'ranked_features': 226,
  'constant_features_removed': ['missing_was_drafted', 'missing_is_rookie'],
  'columns_with_missing_values': 0,
  'maximum_missing_rate': 0.0,
  'finite_after_imputation': True,
  'duplicate_keys': 0,
  'missing_targets': 0,
  'target_min': -4.1,
  'target_median': 2.2,
  'target_max': 471.2,
  'zero_target_rate': 0.45768025078369906,
  'positions': {'QB': 1163, 'RB': 2313, 'TE': 1758, 'WR': 3379}},
 'injury': {'rows': 9077,
  'input_features': 500,
  'ranked_features': 493,
  'constant_features_removed': ['missing_was_drafted',
   'missing_is_rookie',
   'missing_team_changed',
   'lag1_fg_missed_0_19',
   'lag2_fg_missed_0_19',
   'lag3_fg_missed_0_19',
   'lag4_fg_missed_0_19'],
  'columns_with_missing_values': 0,
  'maximum_missing_rate': 0.0,
  'finite_after_imputation': True,
  'duplicate_keys': 0,
  'missing_targets': 0,
  'positive_rate': 0.36785281480665416,
  'class_counts': {0: 5738, 1: 3339},
  'p

## Results

### 3. Rank points features independently

The primary score is the package’s automatic probe-weighted reciprocal-rank consensus across all five methods.

In [4]:
def build_consensus(result, prefix: str) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, object]]:
    primary = voting(result, weights="auto", method="reciprocal_rank").rename(
        columns={"score": "consensus_score"}
    )
    primary.insert(0, "consensus_rank", np.arange(1, len(primary) + 1))
    ranks = result.rank_matrix().rename_axis("feature").reset_index()
    ranks = ranks.rename(columns={method: f"rank_{method}" for method in result.methods})
    raw_scores = result.score_matrix().rename_axis("feature").reset_index()
    raw_scores = raw_scores.rename(columns={method: f"score_{method}" for method in result.methods})
    combined = primary.merge(ranks, on="feature", validate="one_to_one").merge(
        raw_scores, on="feature", validate="one_to_one"
    )
    combined["methods_top_10"] = sum(combined[f"rank_{method}"].le(10) for method in result.methods)
    combined["methods_top_25"] = sum(combined[f"rank_{method}"].le(25) for method in result.methods)

    probe_table = result.probe_table().rename_axis("method").reset_index()
    equal = voting(result, weights=None, method="reciprocal_rank")
    borda = voting(result, weights="auto", method="borda")
    exponential = voting(result, weights="auto", method="exponential")

    def top_overlap(left: pd.DataFrame, right: pd.DataFrame, top_n: int) -> dict[str, float | int]:
        left_set = set(left.head(top_n)["feature"])
        right_set = set(right.head(top_n)["feature"])
        overlap = len(left_set & right_set)
        return {"overlap": overlap, "jaccard": overlap / len(left_set | right_set)}

    rank_matrix = result.rank_matrix()
    method_correlations = {}
    for left_index, left in enumerate(result.methods):
        for right in result.methods[left_index + 1:]:
            method_correlations[f"{left}_vs_{right}"] = float(
                spearmanr(rank_matrix[left], rank_matrix[right]).statistic
            )

    sensitivity = {
        "top_10_auto_vs_equal": top_overlap(primary, equal, 10),
        "top_20_auto_vs_equal": top_overlap(primary, equal, 20),
        "top_20_reciprocal_vs_borda": top_overlap(primary, borda, 20),
        "top_20_reciprocal_vs_exponential": top_overlap(primary, exponential, 20),
        "method_rank_spearman": method_correlations,
    }
    combined.to_csv(OUTPUT_DIR / f"{prefix}_feature_ranking.csv", index=False)
    probe_table.to_csv(OUTPUT_DIR / f"{prefix}_probe_scores.csv", index=False)
    result.save(OUTPUT_DIR / f"{prefix}_ranking_result.joblib")
    return combined, probe_table, sensitivity


with threadpool_limits(limits=MAX_THREADS):
    points_result = feature_ranking(
        points_X,
        points_y,
        task="regression",
        n_jobs=MAX_THREADS,
        random_state=RANDOM_STATE,
        probe=True,
    )
points_ranking, points_probes, points_sensitivity = build_consensus(points_result, "points")
points_ranking.head(20)

,consensus_rank,feature,consensus_score,rank_rf,rank_xg,rank_mi,rank_f_test,rank_l1,score_rf,score_xg,score_mi,score_f_test,score_l1,methods_top_10,methods_top_25
0,1,lag1_games,1.871643,1.0,2.0,1.0,1.0,1.0,0.102609,0.084377,0.208926,3251.414601,40.139976,5,5
1,2,lag1_receptions,0.914762,2.0,5.0,2.0,2.0,2.0,0.091027,0.033767,0.208210,3184.137979,39.117465,5,5
2,3,lag1_passing_tds,0.660238,5.0,1.0,50.0,13.0,4.0,0.040705,0.088338,0.052903,1498.035425,26.300908,3,4
3,4,lag1_receiving_yards,0.544490,3.0,8.0,3.0,3.0,5.0,0.069365,0.022282,0.203994,3006.653746,21.020420,5,5
4,5,lag1_receiving_first_downs,0.298268,4.0,12.0,5.0,5.0,198.0,0.044926,0.014764,0.192122,2837.201395,0.018236,3,4
5,6,lag1_receiving_10,0.298177,6.0,13.0,4.0,4.0,221.0,0.035045,0.014374,0.195679,2838.022096,0.002900,3,4
6,7,lag1_passing_yards,0.295129,8.0,3.0,42.0,15.0,7.0,0.029783,0.044931,0.060043,1455.028953,15.533267,3,4
7,8,lag1_fumbles_total,0.263680,25.0,16.0,17.0,9.0,3.0,0.007209,0.010270,0.094628,2076.948682,30.384173,2,5
8,9,lag1_passing_10,0.215632,10.0,4.0,101.0,18.0,11.0,0.022156,0.038056,0.035736,1446.525509,8.085268,2,4
9,10,lag1_rushing_10,0.185864,14.0,9.0,30.0,21.0,6.0,0.017713,0.017890,0.072467,1427.770979,20.707222,2,4


### 4. Rank injury-report features independently

The classification run uses the same five methods and aggregation rule, but it never receives the points target.

In [5]:
with threadpool_limits(limits=MAX_THREADS):
    injury_result = feature_ranking(
        injury_X,
        injury_y,
        task="classification",
        n_jobs=MAX_THREADS,
        random_state=RANDOM_STATE,
        probe=True,
    )
injury_ranking, injury_probes, injury_sensitivity = build_consensus(injury_result, "injury")
injury_ranking.head(20)

,consensus_rank,feature,consensus_score,rank_rf,rank_xg,rank_mi,rank_f_test,rank_l1,score_rf,score_xg,score_mi,score_f_test,score_l1,methods_top_10,methods_top_25
0,1,lag1_games,1.576960,1.0,1.0,6.0,1.0,1.0,0.120177,0.058917,0.087722,2036.089625,1694.338785,5,5
1,2,lag1_receiving_yards_after_catch,0.648379,2.0,3.0,4.0,4.0,2.5,0.086245,0.028010,0.091963,1519.643038,1199.500000,5,5
2,3,lag1_receiving_yards,0.527763,6.0,15.0,1.0,6.0,8.5,0.024633,0.007221,0.094162,1433.433458,674.528420,4,5
3,4,lag1_receptions,0.487655,9.0,156.0,2.0,3.0,2.5,0.015237,0.002428,0.092072,1549.352531,1199.500000,4,4
4,5,lag1_offense_snaps,0.369030,7.0,32.0,15.0,2.0,4.0,0.020676,0.005348,0.079536,1564.429889,1069.055500,3,4
5,6,missing_draft_number,0.294927,22.0,2.0,47.0,30.5,6.0,0.007826,0.030744,0.046657,763.096044,849.180968,2,3
6,7,lag1_targets,0.201618,34.0,93.0,3.0,5.0,194.5,0.005202,0.003130,0.092007,1515.739051,14.139330,2,2
7,8,lag1_receiving_16,0.189447,12.0,5.0,11.0,12.0,20.0,0.011298,0.014771,0.084195,1236.381251,239.331715,1,5
8,9,draft_number,0.182574,4.0,6.0,35.0,274.0,31.5,0.036455,0.014073,0.071601,28.890710,169.433878,2,2
9,10,lag1_receiving_10,0.161488,28.0,27.0,5.0,7.0,26.5,0.006384,0.005726,0.087741,1375.942549,190.107938,2,2


### 5. Compare the two top-feature lists

Ranked horizontal bars are appropriate because the task is an ordered category comparison. Consensus scores are unitless voting scores and are not effect sizes.

In [6]:
def plot_top_features(ranking: pd.DataFrame, label: str, path: Path, top_n: int = 15) -> None:
    plotted = ranking.head(top_n).iloc[::-1]
    figure, axis = plt.subplots(figsize=(11, 7.5))
    axis.barh(plotted["feature"], plotted["consensus_score"], color="#31688E")
    axis.set_title(f"{label}: top {top_n} FeatureRanker consensus features", loc="left")
    axis.set_xlabel("Probe-weighted reciprocal-rank consensus score")
    axis.set_ylabel("")
    axis.grid(axis="x", color="#D9DEE5", linewidth=0.8)
    axis.set_axisbelow(True)
    for spine in ("top", "right"):
        axis.spines[spine].set_visible(False)
    figure.tight_layout()
    figure.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(figure)


points_plot = OUTPUT_DIR / "points_top_features.png"
injury_plot = OUTPUT_DIR / "injury_top_features.png"
plot_top_features(points_ranking, "Offense fantasy points", points_plot)
plot_top_features(injury_ranking, "Physical injury-report risk", injury_plot)

top_overlap = sorted(set(points_ranking.head(20).feature) & set(injury_ranking.head(20).feature))
summary = {
    "package_version": featureranker.__version__,
    "random_state": RANDOM_STATE,
    "points": {
        "rows": len(points_X),
        "features": points_X.shape[1],
        "top_20": points_ranking.head(20)["feature"].tolist(),
        "probes": points_probes.to_dict(orient="records"),
        "sensitivity": points_sensitivity,
    },
    "injury": {
        "rows": len(injury_X),
        "features": injury_X.shape[1],
        "positive_rate": float(injury_y.mean()),
        "top_20": injury_ranking.head(20)["feature"].tolist(),
        "probes": injury_probes.to_dict(orient="records"),
        "sensitivity": injury_sensitivity,
    },
    "top_20_exact_feature_overlap": top_overlap,
}
(OUTPUT_DIR / "analysis_summary.json").write_text(
    json.dumps(summary, indent=2, default=float), encoding="utf-8"
)
summary

{'package_version': '3.0.4',
 'random_state': 20260820,
 'points': {'rows': 8613,
  'features': 226,
  'top_20': ['lag1_games',
   'lag1_receptions',
   'lag1_passing_tds',
   'lag1_receiving_yards',
   'lag1_receiving_first_downs',
   'lag1_receiving_10',
   'lag1_passing_yards',
   'lag1_fumbles_total',
   'lag1_passing_10',
   'lag1_rushing_10',
   'lag1_receiving_16',
   'lag1_receiving_20',
   'lag1_receiving_tds',
   'draft_number',
   'lag1_passing_20',
   'was_drafted',
   'lag1_rushing_12',
   'lag2_receiving_yards',
   'lag1_rushing_first_downs',
   'lag2_receptions'],
  'probes': [{'method': 'rf',
    1: 0.27316944954196704,
    2: 0.31861520696013906,
    4: 0.3230478416647343,
    8: 0.4695276225111777,
    16: 0.5203209787558224,
    32: 0.5352741441226417,
    64: 0.5530342901324867,
    'score': 0.4275699333841384,
    'skill': 0.4275699333841384},
   {'method': 'xg',
    1: 0.1464901367656658,
    2: 0.34107512458252015,
    4: 0.34067174539319445,
    8: 0.47910387512

## Takeaways

The primary consensus uses automatic probe-skill weights with reciprocal-rank voting across Random Forest, XGBoost, mutual information, ANOVA F-test, and L1.

For points, the top ten are `lag1_games`, `lag1_receptions`, `lag1_passing_tds`, `lag1_receiving_yards`, `lag1_receiving_first_downs`, `lag1_receiving_10`, `lag1_passing_yards`, `lag1_fumbles_total`, `lag1_passing_10`, and `lag1_rushing_10`.

For injury, the top ten are `lag1_games`, `lag1_receiving_yards_after_catch`, `lag1_receiving_yards`, `lag1_receptions`, `lag1_offense_snaps`, `missing_draft_number`, `lag1_targets`, `lag1_receiving_16`, `draft_number`, and `lag1_receiving_10`.

Automatic and equal weights return identical top-20 sets for both targets. Alternate aggregation has a larger impact, particularly for injury. Exact columns should therefore be interpreted as correlated feature families and validated on chronological player-aware folds before pruning a production model.
